In [26]:
import os
import sys
import socket
import xarray as xr
import numpy as np
import importlib
from concurrent.futures import ProcessPoolExecutor
from functools import partial
import re


In [27]:
def get_python_path():
    hostname = socket.gethostname()                                 # 1. Identify the computer by hostname
    code_locations = {                                              # 2. Set default Python code location based on hostname
        "NECMAC04363461.local": "/Users/kimberly.hyde/Documents/",  # Mac laptop
        "nefscsatdata": "/mnt/EDAB_Archive/",                       # Satdata
        "guihyde": "/mnt/EDAB_Archive/"                             # Kim's Satdata container
    }

    default_code_path = os.path.join(code_locations.get(hostname),"nadata/python/utilities/py")    
    if not os.path.isdir(default_code_path):
        print(f"Directory not found: {default_code_path}")
        return {}
    print(f"Default untilities path: {default_code_path}")
    return default_code_path

def global_import(function_map):
    for module_name, function_names in function_map.items():
        try:
            module = importlib.import_module(module_name)
        except ModuleNotFoundError:
            print(f"❌ Module '{module_name}' could not be imported.")
            continue

        for name in function_names:
            try:
                func = getattr(module, name)
                globals()[name] = func
                print(f"✅ Imported: {name} from {module_name}")
            except AttributeError:
                print(f"⚠ Function '{name}' not found in '{module_name}'")
    

# Get the path to the utilities folder
util_path = get_python_path()

# Add to sys.path
if util_path not in sys.path:
    sys.path.append(util_path)

# Find the default utiltity functions and import into global
from import_utilities import get_pyfile_functions
functions = get_pyfile_functions()
gl = global_import(functions)

# Import local calc functions
calc_functions = get_pyfile_functions(os.getcwd(),"calc")
gl = global_import(calc_functions)



Default untilities path: /Users/kimberly.hyde/Documents/nadata/python/utilities/py
✅ Imported: get_python_dir from import_utilities
✅ Imported: get_pyfile_functions from import_utilities
✅ Imported: import_utility_functions from import_utilities
✅ Imported: dataset_defaults from getfiles
✅ Imported: product_defaults from getfiles
✅ Imported: get_datasets_source from getfiles
✅ Imported: get_dataset_dirs from getfiles
✅ Imported: get_dataset_products from getfiles
✅ Imported: get_prod_files from getfiles
✅ Imported: pixel_area from calc_mapped_pixel_area
✅ Imported: total_annual_pp from calc_annual_pp
✅ Imported: day_length_calculation from calc_day_length
✅ Imported: vgpm_models from calc_primary_production
✅ Imported: daily_vgpm from calc_primary_production


In [ ]:
def daily_vgpm(date, chl, sst, par, lat, lon, output_dir,input_dirs):
    #d8 = str(date).split('T')[0]
    
    # Define the output file and check if exists and is up to date
    output_file = os.path.join(output_dir, f"D_{date}-OCCCI-V6.0-GLOBAL_MAPPED-PPD.nc")
    should_process = True
    
    
    if os.path.exists(output_file):
    #    if not any_input_newer(input_dirs, date, output_file):
    #        expected = chl.sel(time=str(date)).time.dt.floor("D").values
    #    if not missing_dates(output_file, expected):
        print(f"✓ Skipping {date} — output file exists.")
        return
    #        should_process = False
    #if not should_process:

    print(f"The output file is: {output_file}")
    
    # Subset dataset to the specified year
    chl_day = chl.sel(time=str(date))
    sst_day = sst.sel(time=str(date))
    par_day = par.sel(time=str(date))

    # Broadcast DOY and LAT for day length calculation
    doy = chl_day['time'].dt.dayofyear
    doy_3d, lat_3d = xr.broadcast(doy, lat)

    # Calculate the day length
    daylength = day_length_calculation(lat_3d.values, doy_3d.values)
    
    # Convert day_length to 2D array
    daylength = np.tile(daylength.reshape(-1, 1), (1, len(lon)))
    
    # Convert day_length to dataset
    day_length = xr.DataArray(
        data=daylength,
        dims=chl_day.dims,
        coords=chl_day.coords,
        name="day_length"
    )
    
    # Change variables to float32
    chl_day = chl_day.astype("float32")
    sst_day = sst_day.astype("float32")
    par_day = par_day.astype("float32")
    day_length = day_length.astype("float32")
   
    print("Running vgpm_models...")
    pp_eppley, pp_vgpm, kdpar, chl_eu, zeu =vgpm_models(chl_day,sst_day,par_day,day_length)
    
    
    #Create data arrays for each variable
    xa_pp_eppley = xr.DataArray(pp_eppley,dims=chl_day.dims,coords=chl_day.coords,name="pp_eppley")
    xa_pp_vgpm = xr.DataArray(pp_vgpm,dims=chl_day.dims,coords=chl_day.coords,name="pp_vgpm")
    xa_kdpar = xr.DataArray(kdpar,dims=chl_day.dims,coords=chl_day.coords,name="pp_kdpar")
    xa_chl_euphotic = xr.DataArray(chl_eu,dims=chl_day.dims,coords=chl_day.coords,name="chl_euphotic")
    xa_zeu = xr.DataArray(zeu,dims=chl_day.dims,coords=chl_day.coords,name="zeu")
    
    
    # Create a dataset
    pp_dataset = xr.Dataset({
        "pp_eppley": xa_pp_eppley,
        "pp_vgpm": xa_pp_vgpm,
        "kdpar": xa_kdpar,
        "chl_euphotic": xa_chl_euphotic,
        "zeu": xa_zeu
    })
    

    pp_dataset = pp_dataset.expand_dims({"time":[date]})
    
    print(f"✓ Writing {output_file}")
    pp_dataset.to_netcdf(output_file)

In [29]:
def load_daily_inputs(date, chl_path, sst_path, par_path):
    chl = xr.open_dataset(chl_path, chunks={"lat": 100, "lon": 100})
    sst = xr.open_dataset(sst_path, chunks={"lat": 100, "lon": 100})
    par = xr.open_dataset(par_path, chunks={"lat": 100, "lon": 100})
    return chl, sst, par


def process_one_date(date, paths, lat, lon, output_dir, input_dirs):
    chl_path, sst_path, par_path = paths
    chl, sst, par = load_daily_inputs(date, chl_path, sst_path, par_path)
    daily_vgpm(date, chl, sst, par, lat, lon, output_dir, input_dirs)

def run_vgpm_batch(date_paths_map, lat, lon, output_dir, input_dirs):
    with ProcessPoolExecutor() as executor:
        futures = []
        for date, paths in date_paths_map.items():
            futures.append(executor.submit(
                process_one_date, date, paths, lat, lon, output_dir, input_dirs
            ))
        for f in futures:
            f.result()  # Optional: catch exceptions

def build_pp_date_map(dates=None, get_date_prod="CHL", chl_dataset=None, sst_dataset=None, par_dataset=None):
    """
    Constructs a date→(CHL, SST, PAR) file path mapping using get_prod_files.

    Parameters:
        dates (list of str): List of dates (YYYYMMDD) to process.
        prod_chl, prod_sst, prod_par (str): Product identifiers for each input variable.

    Returns:
        dict: Mapping of date → (chl_path, sst_path, par_path)
    """
    
    if not dates:
        dates = get_dates_from_filenames('CHL')
    
    date_paths_map = {}

    for date in dates:
        chl_files = get_prod_files("CHL", dataset=chl_dataset, period=date)
        sst_files = get_prod_files("SST", dataset=sst_dataset, period=date)
        par_files = get_prod_files("PAR", dataset=par_dataset, period=date)

        if chl_files and sst_files and par_files:
            date_str = f"{date[:4]}-{date[4:6]}-{date[6:]}"
            date_paths_map[date_str] = (chl_files[0], sst_files[0], par_files[0])
        else:
            print(f"⚠ Missing file(s) for {date}: CHL={bool(chl_files)} SST={bool(sst_files)} PAR={bool(par_files)}")

    print(f"📅 Mapped {len(date_paths_map)} dates with complete product files.")
    return date_paths_map

def get_dates_from_filenames(prod,dataset=None):
    chl_files = get_prod_files("CHL",dataset=dataset)
    dates = []
    for f in chl_files:
        # Assumes filename like '20250701_something.nc' or '20250701.nc'
        basename = os.path.basename(f)
        match = re.search(r"(\d{8})", basename)
        if match:
            yyyymmdd = match.group(1)
            dates.append(yyyymmdd)
    return sorted(dates)


In [30]:
date_path = build_pp_date_map(sst_dataset='CORALSST')
date_path

✓ Using default input data directory: [laptop] → /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/
✅ Found path for 'CHL' → /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/OCCCI/V6.0/SOURCE_DATA/MAPPED_4KM_DAILY/CHL
📦 Found 30 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/OCCCI/V6.0/SOURCE_DATA/MAPPED_4KM_DAILY/CHL
✓ Using default input data directory: [laptop] → /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/
✅ Found path for 'CHL' → /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/OCCCI/V6.0/SOURCE_DATA/MAPPED_4KM_DAILY/CHL
📦 Found 30 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/OCCCI/V6.0/SOURCE_DATA/MAPPED_4KM_DAILY/CHL
✓ Using default input data directory: [laptop] → /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/
⚠ No 'OUTPUT' path found for 'CORALSST' – proceeding with SOURCE only.
✅ Found path for 'SST' → /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/CORALSST/V3.1/SOURCE_DATA/MAPPED_5KM_DAILY/SST

{'1998-01-01': ('/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/OCCCI/V6.0/SOURCE_DATA/MAPPED_4KM_DAILY/CHL/ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-19980129-fv6.0.nc',
  '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/CORALSST/V3.1/SOURCE_DATA/MAPPED_5KM_DAILY/SST/coraltemp_v3.1_19980116.nc',
  '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/GLOBCOLOUR/V4.2.1/SOURCE_DATA/MAPPED_4KM_DAILY/PAR/L3m_19980119__GLOB_4_AV-SWF_PAR_DAY_00.nc'),
 '1998-01-02': ('/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/OCCCI/V6.0/SOURCE_DATA/MAPPED_4KM_DAILY/CHL/ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-19980129-fv6.0.nc',
  '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/CORALSST/V3.1/SOURCE_DATA/MAPPED_5KM_DAILY/SST/coraltemp_v3.1_19980116.nc',
  '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/GLOBCOLOUR/V4.2.1/SOURCE_DATA/MAPPED_4KM_DAILY/PAR/L3m_19980119__GLOB_4_AV-SWF_PAR_DAY_00.nc'),
 '1998-01-03': ('/Users/kimberly.hyde/Documents/na

In [ ]:
chl_path = get_prod_files('chl',dataset='OCCCI',getfilepath=True)
sst_path = get_prod_files('sst',dataset='CORALSST',getfilepath=True)
par_path = get_prod_files('par',dataset='GLOBCOLOUR',getfilepath=True)

In [25]:
get_prod_files('par')

✓ Using default input data directory: [laptop] → /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/
⚠ No 'OUTPUT' path found for 'GLOBCOLOUR' – proceeding with SOURCE only.
✅ Found path for 'PAR' → /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/GLOBCOLOUR/V4.2.1/SOURCE_DATA/MAPPED_4KM_DAILY/PAR
📦 Found 31 .nc files in: /Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/GLOBCOLOUR/V4.2.1/SOURCE_DATA/MAPPED_4KM_DAILY/PAR


['/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/GLOBCOLOUR/V4.2.1/SOURCE_DATA/MAPPED_4KM_DAILY/PAR/L3m_19980119__GLOB_4_AV-SWF_PAR_DAY_00.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/GLOBCOLOUR/V4.2.1/SOURCE_DATA/MAPPED_4KM_DAILY/PAR/L3m_19980117__GLOB_4_AV-SWF_PAR_DAY_00.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/GLOBCOLOUR/V4.2.1/SOURCE_DATA/MAPPED_4KM_DAILY/PAR/L3m_19980102__GLOB_4_AV-SWF_PAR_DAY_00.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/GLOBCOLOUR/V4.2.1/SOURCE_DATA/MAPPED_4KM_DAILY/PAR/L3m_19980126__GLOB_4_AV-SWF_PAR_DAY_00.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/GLOBCOLOUR/V4.2.1/SOURCE_DATA/MAPPED_4KM_DAILY/PAR/L3m_19980128__GLOB_4_AV-SWF_PAR_DAY_00.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/GLOBCOLOUR/V4.2.1/SOURCE_DATA/MAPPED_4KM_DAILY/PAR/L3m_19980101__GLOB_4_AV-SWF_PAR_DAY_00.nc',
 '/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/GLOBCOLOUR/V4.2.1/SOURCE_DATA/MAPPED_4

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from functools import partial

def process_one_date(date, paths, lat, lon, output_dir, input_dirs):
    chl_path, sst_path, par_path = paths
    chl, sst, par = load_daily_inputs(date, chl_path, sst_path, par_path)
    daily_vgpm(date, chl, sst, par, lat, lon, output_dir, input_dirs)


In [ ]:
def run_vgpm_batch(date_paths_map, lat, lon, output_dir, input_dirs):
    with ProcessPoolExecutor() as executor:
        futures = []
        for date, paths in date_paths_map.items():
            futures.append(executor.submit(
                process_one_date, date, paths, lat, lon, output_dir, input_dirs
            ))
        for f in futures:
            f.result()  # Optional: catch exceptions


In [ ]:
# Load your dataset (must have lat/lon as 2D arrays if irregular)
dirdata = r'/Users/kimberly.hyde/Documents/nadata/DATASETS_SOURCE/'
dirchl = os.path.join(dirdata,'OCCCI/V6.0/SOURCE_DATA/MAPPED_4KM_DAILY/CHL/')
ds = xr.open_mfdataset(os.path.join(dirchl,'*.nc'), combine='by_coords')

area_da = calc_mapped_pixel_area(ds['lat'].values, ds['lon'].values)